# الدرس التاسع: برمجيات وسيطة لتلخيص المحادثة وإدارة الذاكرة (Summarization Middleware)

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعرف على احدى اقوى الاضافات في بنية وكلاء LangChain 1.x: برمجية `SummarizationMiddleware` المدمجة ضمن `langchain.agents.middleware`.

## التحدي: انفجار نافذة السياق (Context Window Explosion)
عندما يخوض الوكيل محادثة طويلة تمتد لعشرات الرسائل واستدعاءات الادوات:
1. تزداد تكلفة استهلاك التوكنات (Cost per request) بشكل تصاعدي.
2. تزداد ازمنة الاستجابة (Latency).
3. قد تتجاوز المحادثة الحد الاقصى المسموح به لنافذة السياق (Context Window Limit) مما يسبب توقف النظام.

## الحل الحديث: `SummarizationMiddleware`
تعمل هذه البرمجية كوسيط ذكي (Middleware) قبل استدعاء النموذج:
- تراقب طول سجل الرسائل بناء على شرط تفعيل محدد (`trigger`) مثل عدد الرسائل او التوكنات.
- عندما يتحقق الشرط، تقوم البرمجية بتلخيص الرسائل القديمة واستبدالها بسياق ملخص منظم يوضح (Session Intent, Summary, Artifacts, Next Steps).
- تحتفظ فقط بعدد محدد من احدث الرسائل (`keep`) لضمان استمرارية السياق اللحظي دون انقطاع.

## الخطوة 1: استيراد البرمجيات الوسيطة وتجهيز البيئة
نقوم باستيراد `SummarizationMiddleware` من حزمة البرمجيات الوسيطة الحديثة `langchain.agents.middleware`.

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq", temperature=0)

## الخطوة 2: تهيئة اداة للمحادثة وتكوين الوسيط (Middleware Configuration)
نقوم باعداد اداة مساعدة لحساب الاسعار، وننشئ كائن `SummarizationMiddleware`.
نضبط شرط التفعيل ليتم التلخيص عند تجاوز 4 رسائل مع الابقاء على اخر رسالتين فقط.

In [ ]:
def calculate_discount(price: float, discount_percent: float) -> str:
    """Calculate final price after applying discount percentage."""
    final_price = price * (1 - (discount_percent / 100))
    return f"Original: ${price:.2f}, Discount: {discount_percent}%, Final: ${final_price:.2f}"

# تهيئة وسيط التلخيص التلقائي
summarizer_middleware = SummarizationMiddleware(
    model=model,
    trigger=("messages", 4),  # يتم تفعيل التلخيص عند الوصول لـ 4 رسائل
    keep=("messages", 2)      # الاحتفاظ بأحدث رسالتين فقط
)

print("SummarizationMiddleware successfully configured.")

## الخطوة 3: بناء الوكيل المجهز بالبرمجية الوسيطة عبر `create_agent`
نمرر كائن `SummarizationMiddleware` ضمن قائمة `middleware` في دالة `create_agent`.

In [ ]:
agent = create_agent(
    model=model,
    tools=[calculate_discount],
    system_prompt="You are a polite shopping assistant. Help customers calculate discounts and remember their preferences.",
    middleware=[summarizer_middleware]
)

print("Agent compiled with Middleware support:", type(agent))

## الخطوة 4: محاكاة جلسة محادثة طويلة ومراقبة سلوك التلخيص
نقوم بارسال عدة رسائل متعاقبة لرؤية كيف يتعامل الوكيل مع سياق المحادثة الممتد.

In [ ]:
# المحادثة الاولى
response1 = agent.invoke({
    "messages": [
        ("user", "Hello! My name is Jordan and my budget is $500.")
    ]
})
print("Turn 1 Response:")
print(response1["messages"][-1].content)

# المحادثة الثانية مع استدعاء الاداة
response2 = agent.invoke({
    "messages": response1["messages"] + [
        ("user", "I want to buy a laptop for $400 with a 15% discount. What is the final price?")
    ]
})
print("\nTurn 2 Response:")
print(response2["messages"][-1].content)

# المحادثة الثالثة (يتجاوز الحد المسموح وتبدأ البرمجية الوسيطة بدمج الملخص)
response3 = agent.invoke({
    "messages": response2["messages"] + [
        ("user", "What is my name and do I have enough budget left for it?")
    ]
})
print("\nTurn 3 Response (Context retained via Middleware):")
print(response3["messages"][-1].content)

## الخطوة 5: فحص الرسائل للتأكد من حفظ السياق
نطبع سجل الرسائل الاخير لنلاحظ كيف احتفظ الوكيل بالمعلومات الجوهرية (الاسم والميزانية) بنجاح.

In [ ]:
print("Total messages in active state:", len(response3["messages"]))
for i, m in enumerate(response3["messages"]):
    preview = str(m.content)[:100]
    print(f"Message {i+1} [{m.type}]: {preview}...")